# Phase 0：問卷 Adapter — 三大信號萃取

從 SQL Server `Quiz..OESTE/OESTA/OESTB` 萃取 82,105 筆問卷資料，
輸出可接入 Phase 3 痛需累積表的結構化信號。

| 信號 | 對應 L 層 | 來源題型 | 萃取方式 |
|------|---------|---------|---------|
| 企業痛點 (pain_point) | L1–L4 | 多選/問答 | 規則詞典 + LLM |
| 決策角色 (role) | L2 | 單選/多選 | 規則映射 |
| 決策因素 (decision_factor) | L5/L6 | 單選/多選/問答 | 規則詞典 + LLM |

**輸出**：`survey_signals.jsonl`（含 company_id，可接 Phase 3 痛需累積表）


## 0. 安裝套件

In [ ]:
!pip install google-genai pyodbc pandas tqdm

## 1. 匯入套件 & 全域設定

In [ ]:
import os, re, json, time, hashlib, configparser
from pathlib import Path
from datetime import datetime
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd

# ── 路徑設定 ──────────────────────────────────────────
BASE_DIR   = Path(r"D:\yujui\痛點需求地圖")
OUTPUT_DIR = BASE_DIR / "survey_adapter_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULT_JSONL    = OUTPUT_DIR / "survey_signals.jsonl"
LLM_CACHE_JSONL = OUTPUT_DIR / "_llm_cache.jsonl"      # 問答題 LLM 結果快取
REPORT_CSV      = OUTPUT_DIR / "survey_adapter_report.csv"

# ── LLM 設定 ──────────────────────────────────────────
MODEL          = "gemini-2.0-flash"
TEMP           = 0.1        # 低溫，穩定輸出
RPM_LIMIT      = 14
SLEEP_PER_CALL = 60 / RPM_LIMIT

# ── 題型常數 ──────────────────────────────────────────
TYPE_SINGLE = "單選"
TYPE_MULTI  = "多選"
TYPE_OPEN   = "問答"

LAYERS = ["L1","L2","L3","L4","L5","L6","L7"]

print("全域設定完成")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}")


## 2. 載入設定 & 建立連線

In [ ]:
import pyodbc

# ── SQL Server ────────────────────────────────────────
SQL_INI = r"D:\yujui\SqlServer18.txt"
cfg = configparser.ConfigParser()
cfg.read(SQL_INI, encoding="utf-8")
sec = cfg["mssql"]
conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};SERVER={sec['server']};DATABASE=DSC_CRM_SP;"
    f"UID={sec['uid']};PWD={sec['pwd']};",
    autocommit=True,
)
print("SQL Server 連線成功")
# Gemini 不需要（問卷 Adapter 全程複用 Step 0 結果，不呼叫 LLM）


---
## 子任務 A：資料撈取

從 SQL Server 撈取全量問卷作答，補入 `company_id`（潛客代號）與 `ym`（活動年月）。
題型欄位：`單選`、`多選`、`問答`，對應後續分流處理。


In [ ]:
# ── A1：Schema 確認（快查欄位，確認無誤後可跳過）──────
cursor = conn.cursor()

# 快查題型分布（確認資料庫結構正常）
cursor.execute("""
    SELECT TOP 5 OESTA001, OESTA005, OESTA009
    FROM Quiz..OESTA
    WHERE OESTA009 IS NOT NULL
""")
print("OESTA 樣本（問項代號 / 題型代碼 / 問項文字）：")
for r in cursor.fetchall():
    print(" ", r)

cursor.execute("""
    SELECT
        COUNT(*) AS 總筆數,
        SUM(CASE WHEN OESTE004='2' THEN 1 ELSE 0 END) AS 單選,
        SUM(CASE WHEN OESTE004='3' THEN 1 ELSE 0 END) AS 多選,
        SUM(CASE WHEN OESTE004='4' THEN 1 ELSE 0 END) AS 問答
    FROM Quiz..OESTE
    WHERE OESTE040 IS NOT NULL
""")
row = cursor.fetchone()
print(f"\nOESTE 題型統計：總={row[0]:,}  單選={row[1]:,}  多選={row[2]:,}  問答={row[3]:,}")


In [ ]:
# ── A2：撈取全量問卷資料（含 company_id）──────────────
# STRING_SPLIT 需要 SQL Server 2016+，改為 Python 側拆分多選答案

SURVEY_SQL = """
SELECT DISTINCT
     [潛客代號]   = ISNULL([E].[潛客代號], '')
    ,[company_id] = ISNULL([IG].IG018, '')
    ,[ym]         = FORMAT(TRY_CONVERT(DATE, [E].[活動日期]), 'yyyy-MM')
    ,[活動名稱]   = ISNULL(IA.IA003, '')
    ,[問項]       = [A].[問項]
    ,[問項類型]   = [A].[問項類型]
    ,[答案原始]   = [E].[答案]
    ,[選項1]  = [A].OESTA012, [選項2]  = [A].OESTA013
    ,[選項3]  = [A].OESTA014, [選項4]  = [A].OESTA015
    ,[選項5]  = [A].OESTA016, [選項6]  = [A].OESTA017
    ,[選項7]  = [A].OESTA020, [選項8]  = [A].OESTA021
    ,[選項9]  = [A].OESTA022, [選項10] = [A].OESTA023
    ,[選項11] = [A].OESTA024, [選項12] = [A].OESTA025
FROM (
    SELECT [活動代號]=OESTB013,[問卷代號]=OESTB001
    FROM Quiz..OESTB
) [B]
LEFT JOIN (
    SELECT [問卷代號]=OESTA019,[問項代號]=OESTA001,
        [問項類型]=(CASE OESTA005 WHEN '2' THEN '單選' WHEN '3' THEN '多選' WHEN '4' THEN '問答' END),
        [問項]=OESTA009,
        OESTA012, OESTA013, OESTA014, OESTA015,
        OESTA016, OESTA017, OESTA020, OESTA021,
        OESTA022, OESTA023, OESTA024, OESTA025
    FROM Quiz..OESTA
) [A] ON [A].[問卷代號] = [B].[問卷代號]
LEFT JOIN (
    SELECT [問卷代號]=OESTE001,[問項代號]=OESTE003,
        [客戶姓名]=OESTE041,[潛客代號]=OESTE040,
        [活動日期]=OESTE042,[場次]=OESTE043,
        [問項類型]=(CASE OESTE004 WHEN '2' THEN '單選' WHEN '3' THEN '多選' WHEN '4' THEN '問答' END),
        [答案]=(CASE
            WHEN OESTE004='2' AND ISNULL(OESTE017,'')!='' THEN ISNULL(OESTE017,'')
            WHEN OESTE004='3' AND ISNULL(OESTE018,'')!='' THEN ISNULL(OESTE018,'')
            WHEN OESTE004='4' AND ISNULL(OESTE019,'')!='' THEN ISNULL(OESTE019,'')
            ELSE NULL END)
    FROM Quiz..OESTE
    WHERE OESTE040 IS NOT NULL
) [E] ON [E].[問卷代號]=[B].[問卷代號] AND [E].[問項代號]=[A].[問項代號]
LEFT JOIN [DSC_CRM_SP].[dbo].CRMIA IA ON IA.IA001 = [B].[活動代號]
LEFT JOIN [DSC_CRM_SP].[dbo].CRMIG IG ON IG.IG040 = [E].[潛客代號]
WHERE [E].[答案] IS NOT NULL
  AND LEN(LTRIM(RTRIM([E].[答案]))) > 0
  AND [A].[問項類型] IS NOT NULL
"""

print("撈取中（約 82k 筆）...")
survey_raw_base = pd.read_sql(SURVEY_SQL, conn)
survey_raw_base = survey_raw_base[
    survey_raw_base["答案原始"].notna() &
    (survey_raw_base["答案原始"].str.strip() != "")
].copy()
print(f"SQL 撈取完成：{len(survey_raw_base):,} 筆")

# ── Python 側展開多選答案（取代 STRING_SPLIT）──────────────
def expand_multi_answers(df):
    """多選題：答案原始為逗號分隔的選項編號（如 '1,3,5'），對應選項文字後展開"""
    rows = []
    for _, row in df.iterrows():
        q_type = row["問項類型"]
        raw    = str(row["答案原始"]).strip()

        if q_type in ("單選", "問答"):
            # 單選/問答：直接用原始答案（單選為選項編號，問答為文字）
            if q_type == "單選":
                col = f"選項{raw}"
                val = row.get(col, "")
                ans = str(val).strip() if pd.notna(val) and str(val).strip() else raw
            else:
                ans = raw
            r = row.to_dict()
            r["答案"] = ans
            rows.append(r)
        else:
            # 多選：逗號分隔選項編號，各自對應選項文字
            nums = [n.strip() for n in raw.split(",") if n.strip()]
            for n in nums:
                col = f"選項{n}"
                val = row.get(col, "")
                ans = str(val).strip() if pd.notna(val) and str(val).strip() else n
                r = row.to_dict()
                r["答案"] = ans
                rows.append(r)
    return pd.DataFrame(rows)

survey_raw = expand_multi_answers(survey_raw_base)
# 去掉選項欄位（不再需要）
opt_cols = [f"選項{i}" for i in list(range(1,13))]
survey_raw = survey_raw.drop(columns=[c for c in opt_cols if c in survey_raw.columns])
survey_raw = survey_raw[survey_raw["答案"].str.strip() != ""].copy()

# answer_id：唯一識別（潛客代號 + 問項 + 答案）
survey_raw["answer_id"] = (
    survey_raw["潛客代號"] + "||" + survey_raw["問項"] + "||" + survey_raw["答案"]
).apply(lambda t: hashlib.sha256(t.encode()).hexdigest()[:16])

print(f"展開後：{len(survey_raw):,} 筆")
print(survey_raw["問項類型"].value_counts().to_string())
print(survey_raw.head(2)[["潛客代號","問項類型","問項","答案"]].to_string())


---
## 子任務 B：規則萃取（多選 / 單選）

針對**有明確選項**的題目，用詞典直接映射到三大信號，**不呼叫 LLM，零費用**。

### 三大信號判斷依據

| 問項關鍵詞 | 信號類型 | L 層 |
|---------|---------|------|
| 挑戰、困境、問題、風險、障礙、不足 | pain_point | L1–L3 |
| 趨勢、政策、法規、競爭、ESG | pain_point | L4 |
| 角色、職位、決策者、負責人、評估單位 | role | L2 |
| 評估因素、關注重點、考量、選擇依據 | decision_factor | L5 |
| 滿意度、態度、優先、比較 | decision_factor | L6 |


In [ ]:
# ── B1：問項關鍵詞詞典 → 信號類型 & L 層 ──────────────

# 問項文字 → (signal_type, l_layer)
QUESTION_RULES = [
    # ── 痛點信號 L1：起因/阻礙 ──
    (["挑戰","困境","問題","阻礙","障礙","瓶頸","困難","跟不上","不足","缺乏"],
     "pain_point", "L1"),
    # ── 痛點信號 L2：角色壓力 ──
    (["人員","人才","培訓","員工","跟不上","競爭"],
     "pain_point", "L2"),
    # ── 痛點信號 L3：戰略目標 ──
    (["成本","競爭力","交期","降本","轉型","效率","生產力","目標"],
     "pain_point", "L3"),
    # ── 痛點信號 L4：產業議題 ──
    (["趨勢","法規","政策","ESG","供應鏈","永續","碳","競爭對手"],
     "pain_point", "L4"),
    # ── 角色信號 L2 ──
    (["角色","職位","負責","決策","主管","單位","評估單位","關鍵人"],
     "role", "L2"),
    # ── 決策因素 L5：評估/方案選擇 ──
    (["評估","考量","選擇","依據","因素","關注","重視","功能","需求"],
     "decision_factor", "L5"),
    # ── 決策因素 L6：態度/優先 ──
    (["滿意","態度","優先","緊急","迫切","比較","變化"],
     "decision_factor", "L6"),
]

def match_question_rule(question_text: str):
    """回傳 (signal_type, l_layer) 或 None"""
    q = str(question_text)
    for keywords, sig_type, layer in QUESTION_RULES:
        if any(kw in q for kw in keywords):
            return sig_type, layer
    return None, None

# 統計規則覆蓋率
rule_hits = survey_raw["問項"].apply(lambda q: match_question_rule(q)[0] is not None)
print(f"規則可覆蓋（多選+單選）：{rule_hits.sum():,} 筆 ({rule_hits.mean()*100:.1f}%)")
print(f"規則未覆蓋：{(~rule_hits).sum():,} 筆（其中問答題送 LLM）")


In [ ]:
# ── B2：規則萃取執行（多選 + 單選）────────────────────
rule_records = []

for _, row in survey_raw[survey_raw["問項類型"].isin(["多選", "單選"])].iterrows():
    sig_type, layer = match_question_rule(row["問項"])
    if sig_type is None:
        continue
    answer = str(row["答案"]).strip()
    if not answer or answer in ("無", "N/A", "不適用", "其他", "1", "2", "3"):
        continue

    rule_records.append({
        "signal_id":      row["answer_id"],
        "company_id":     row.get("潛客代號", ""),
        "ym":             row.get("ym", ""),
        "活動名稱":       row.get("活動名稱", ""),
        "signal_type":    sig_type,
        "l_layer":        layer,
        "signal_content": answer,
        "question_text":  row["問項"],
        "question_type":  row["問項類型"],
        "confidence":     1.0,
        "extract_method": "rule",
    })

rule_df = pd.DataFrame(rule_records)
print(f"規則萃取完成：{len(rule_df):,} 筆")
print(rule_df.groupby(["signal_type","l_layer"]).size().sort_index().to_string())


---
## 子任務 C：問答題轉換（複用 Step 0 survey_labeled.jsonl）

Step 0 Task D 已完成問答題 LLM 標注，直接讀取 `survey_labeled.jsonl`，
**不重跑 LLM，零費用**。

| Step 0 格式 | Adapter 格式 |
|------------|-------------|
| qa_id（問項+答案 hash） | signal_id |
| l_layer（L1–L7） | → LAYER_TO_SIGNAL 映射 signal_type |
| confidence | 保留 |
| 無 company_id | join survey_raw 補入（1 qa_id 對多公司） |


In [ ]:
# ── C1：載入 Step 0 survey_labeled.jsonl（不重跑 LLM）────
# Step 0 Task D 已完成問答題標注，直接複用

SURVEY_LABELED_JSONL = Path(r"D:\yujui\痛點需求地圖\seed_output\survey_labeled.jsonl")
OPEN_CONF_THRESH = 0.75   # 與 Step 0 一致

# l_layer → signal_type 映射
LAYER_TO_SIGNAL = {
    "L1": "pain_point",    # 起因/阻礙 → 痛點
    "L2": "pain_point",    # 角色壓力 → 痛點（問卷語境多為組織困境）
    "L3": "pain_point",    # 戰略目標 → 痛點
    "L4": "pain_point",    # 產業議題 → 痛點
    "L5": "decision_factor",  # 問項精煉 → 決策因素
    "L6": "decision_factor",  # 動態屬性 → 決策因素
    "L7": "decision_factor",  # 實戰對策 → 決策因素
}

if not SURVEY_LABELED_JSONL.exists():
    print(f"⚠️  找不到 {SURVEY_LABELED_JSONL}")
    print("請先執行 L1L7種子庫建立.ipynb 的子任務 D")
else:
    labeled = []
    with open(SURVEY_LABELED_JSONL, encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            labeled.append(json.loads(line))

    labeled_df = pd.DataFrame(labeled)
    print(f"survey_labeled.jsonl 載入：{len(labeled_df):,} 筆")
    print(f"各層分布：\n{labeled_df['l_layer'].value_counts().sort_index().to_string()}")

    valid_labeled = labeled_df[
        (labeled_df["confidence"] >= OPEN_CONF_THRESH) &
        (labeled_df["l_layer"].isin(LAYERS))
    ].copy()
    print(f"\n高信心（≥{OPEN_CONF_THRESH}）：{len(valid_labeled):,} 筆")


In [ ]:
# ── C2：join company_id，展開為每公司一筆信號 ─────────────
# qa_id（問項+答案）對應多個公司：同一個問答組合可來自多位受訪者
# 用 survey_raw 建立 qa_id → (潛客代號, ym) 對照

# 補 qa_id 到 survey_raw（與 Step 0 相同公式：問項||答案）
survey_raw["qa_id"] = (survey_raw["問項"] + "||" + survey_raw["答案"]).apply(
    lambda t: hashlib.sha256(t.encode()).hexdigest()[:16]
)

# 有效 qa_ids（Step 0 高信心）
valid_qa_ids = set(valid_labeled["qa_id"])

# 每個 qa_id 對應的公司清單
qa_companies = (
    survey_raw[survey_raw["qa_id"].isin(valid_qa_ids)]
    [["qa_id","潛客代號","ym","活動名稱"]]
    .drop_duplicates()
)
print(f"匹配到 qa_id：{qa_companies['qa_id'].nunique():,} 個")
print(f"展開（含各公司）：{len(qa_companies):,} 筆")

# join：labeled × companies（一題多公司展開）
merged = valid_labeled.merge(qa_companies, on="qa_id", how="left")

open_records = []
for _, row in merged.iterrows():
    open_records.append({
        "signal_id":      row["qa_id"] + "_" + str(row.get("潛客代號",""))[:8],
        "company_id":     row.get("潛客代號", ""),
        "ym":             row.get("ym", ""),
        "活動名稱":       row.get("活動名稱", ""),
        "signal_type":    LAYER_TO_SIGNAL.get(row["l_layer"], "pain_point"),
        "l_layer":        row["l_layer"],
        "signal_content": str(row.get("答案", ""))[:100],
        "question_text":  row.get("問項", ""),
        "question_type":  TYPE_OPEN,
        "confidence":     row["confidence"],
        "extract_method": "step0_llm",
    })

open_df_signals = pd.DataFrame(open_records)
print(f"\n問答題信號：{len(open_df_signals):,} 筆")
print(f"有 company_id：{(open_df_signals['company_id']!='').sum():,} 筆")
print(f"無 company_id（未匹配）：{(open_df_signals['company_id']=='').sum():,} 筆")


---
## 子任務 D：合併 & 輸出


In [12]:
# ── D1：合併規則結果 + Step0 LLM 結果，輸出 survey_signals.jsonl ──
# rule_df        來自 B2（多選/單選規則萃取）
# open_df_signals 來自 C2（問答題，複用 survey_labeled.jsonl）

all_signals = pd.concat([rule_df, open_df_signals], ignore_index=True)
all_signals = all_signals.drop_duplicates(subset=["signal_id"])
all_signals = all_signals[all_signals["signal_type"] != "none"].copy()

print(f"合併後總信號：{len(all_signals):,} 筆")
print(f"  規則萃取：{len(rule_df):,} 筆")
print(f"  Step0 LLM：{len(open_df_signals):,} 筆")

all_signals.to_json(RESULT_JSONL, orient="records", lines=True, force_ascii=False)
print(f"\n輸出：{RESULT_JSONL}")


合併後總信號：1,024,560 筆
  規則萃取：527,709 筆
  Step0 LLM：1,611,260 筆

輸出：D:\yujui\痛點需求地圖\survey_adapter_output\survey_signals.jsonl


In [13]:
# ── D2：by-company 匯總（Phase 3 痛需累積表前置）───────
COMPANY_CSV = OUTPUT_DIR / "survey_by_company.csv"

company_signals = (
    all_signals[all_signals["company_id"] != ""]
    .groupby(["company_id", "signal_type", "l_layer"])
    .agg(
        count=("signal_id", "count"),
        latest_ym=("ym", "max"),
        sample_content=("signal_content", lambda x: x.iloc[0] if len(x) > 0 else ""),
    )
    .reset_index()
)
company_signals.to_csv(COMPANY_CSV, index=False, encoding="utf-8-sig")

print(f"by-company 匯總：{len(company_signals):,} 筆  |  公司數：{company_signals['company_id'].nunique():,}")
print(f"輸出：{COMPANY_CSV}")


by-company 匯總：75,213 筆  |  公司數：15,818
輸出：D:\yujui\痛點需求地圖\survey_adapter_output\survey_by_company.csv


---
## 子任務 E：報告


In [14]:
# ── E1：三大信號統計 ─────────────────────────────────────
total = len(all_signals)
print(f"問卷 Adapter 完成  共 {total:,} 筆有效信號\n")

print("── 信號類型分布 ─────────────────────────")
for st in ["pain_point","role","decision_factor"]:
    n = (all_signals["signal_type"] == st).sum()
    print(f"  {st:<20s} {n:>7,} 筆  ({n/total*100:.1f}%)")

print("\n── 各 L 層分布 ──────────────────────────")
for layer in LAYERS:
    n = (all_signals["l_layer"] == layer).sum()
    print(f"  {layer}  {n:>7,} 筆")

print("\n── 萃取方式 ─────────────────────────────")
for m in ["rule","llm"]:
    n = (all_signals["extract_method"] == m).sum()
    print(f"  {m:<6s}  {n:>7,} 筆  ({n/total*100:.1f}%)")

print("\n── 公司覆蓋 ─────────────────────────────")
n_company = all_signals[all_signals["company_id"] != ""]["company_id"].nunique()
print(f"  有 company_id 的信號筆數：{(all_signals['company_id'] != '').sum():,}")
print(f"  覆蓋公司數：{n_company:,}")

print(f"\n輸出路徑：")
print(f"  {RESULT_JSONL}  （全量信號）")
print(f"  {COMPANY_CSV}   （by-company 匯總）")


問卷 Adapter 完成  共 1,024,560 筆有效信號

── 信號類型分布 ─────────────────────────
  pain_point           520,453 筆  (50.8%)
  role                  25,320 筆  (2.5%)
  decision_factor      478,787 筆  (46.7%)

── 各 L 層分布 ──────────────────────────
  L1  321,550 筆
  L2   60,948 筆
  L3   83,193 筆
  L4   80,082 筆
  L5  376,223 筆
  L6   99,732 筆
  L7    2,832 筆

── 萃取方式 ─────────────────────────────
  rule    359,273 筆  (35.1%)
  llm           0 筆  (0.0%)

── 公司覆蓋 ─────────────────────────────
  有 company_id 的信號筆數：1,023,719
  覆蓋公司數：15,818

輸出路徑：
  D:\yujui\痛點需求地圖\survey_adapter_output\survey_signals.jsonl  （全量信號）
  D:\yujui\痛點需求地圖\survey_adapter_output\survey_by_company.csv   （by-company 匯總）
